# Make 10-minute crypto returns for portfolio risk

This notebook loads 1-minute log prices and the final 200 tickers, converts them to daily 10-minute log returns, and saves files that can be loaded directly in the FIVAR portfolio-risk notebook.

In [ ]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
from pathlib import Path

np.set_printoptions(precision=8, suppress=False, linewidth=120)


In [ ]:
# Paths

BASE_DIR = Path(".")

UNIVERSE_OUTPUT_DIR = BASE_DIR / "crypto_universe_output"
PRVM_OUTPUT_DIR = BASE_DIR / "crypto_prvm_output"
FIVAR_OUTPUT_DIR = BASE_DIR / "crypto_fivar_HAR_output"

LOG_PRICE_PATH = UNIVERSE_OUTPUT_DIR / "log_price_2023_2025_top200.parquet"
TICKER_PATH = UNIVERSE_OUTPUT_DIR / "final_tickers_formation_2022_top200.csv"
PRVM_NPZ_PATH = PRVM_OUTPUT_DIR / "prvm_gamma_jv_2023_2025_top200.npz"

FIVAR_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("LOG_PRICE_PATH:", LOG_PRICE_PATH)
print("TICKER_PATH:", TICKER_PATH)
print("PRVM_NPZ_PATH:", PRVM_NPZ_PATH)
print("FIVAR_OUTPUT_DIR:", FIVAR_OUTPUT_DIR)

In [ ]:
# Load log prices and final ticker order

log_price = pd.read_parquet(LOG_PRICE_PATH)
ticker_df = pd.read_csv(TICKER_PATH)

# ticker column 확인. 사진 기준 column name은 'ticker'
if "ticker" in ticker_df.columns:
    tickers = ticker_df["ticker"].astype(str).tolist()
else:
    tickers = ticker_df.iloc[:, 0].astype(str).tolist()

log_price.index = pd.to_datetime(log_price.index)
log_price = log_price.sort_index()

# PRVM에서 사용한 200개 ticker 순서와 반드시 맞춤
missing = [t for t in tickers if t not in log_price.columns]
if missing:
    raise ValueError(f"Missing tickers in log_price: {missing[:10]} ... total={len(missing)}")

log_price = log_price[tickers].astype(np.float64)

print("log_price shape:", log_price.shape)
print("date range:", log_price.index.min(), log_price.index.max())
print("num tickers:", len(tickers))
print("first tickers:", tickers[:10])
display(log_price.head())
display(ticker_df.head())


log_price shape: (1576800, 200)
date range: 2023-01-01 00:00:00 2025-12-31 23:59:00
num tickers: 200
first tickers: ['BTCUSDT', 'ETHUSDT', 'GMTUSDT', 'BNBUSDT', 'LUNAUSDT', 'SOLUSDT', 'XRPUSDT', 'DOGEUSDT', 'APEUSDT', 'SHIBUSDT']


,BTCUSDT,ETHUSDT,GMTUSDT,BNBUSDT,LUNAUSDT,SOLUSDT,XRPUSDT,DOGEUSDT,APEUSDT,SHIBUSDT,...,VTHOUSDT,ONGUSDT,OXTUSDT,AMPUSDT,AVAUSDT,WANUSDT,ALCXUSDT,DFUSDT,OMUSDT,XNOUSDT
datetime,,,,,,,,,,,,,,,,,,,,,
2023-01-01 00:00:00,9.713758,7.086847,-1.468372,5.506550,0.231905,2.300583,-1.081755,-2.654699,1.291159,-11.724882,...,-7.008681,-1.546933,-2.695628,-5.792614,-0.638659,-1.714798,2.617396,-3.26492,-3.568433,-0.450986
2023-01-01 00:01:00,9.713495,7.086813,-1.468372,5.506550,0.233411,2.303585,-1.081755,-2.654557,1.290609,-11.724882,...,-7.007576,-1.546933,-2.695628,-5.792614,-0.638659,-1.714243,2.617396,-3.26492,-3.568433,-0.450986
2023-01-01 00:02:00,9.713321,7.086613,-1.470111,5.506144,0.231826,2.303585,-1.082641,-2.655837,1.290609,-11.724882,...,-7.007576,-1.546933,-2.695628,-5.792614,-0.638659,-1.712024,2.617396,-3.26492,-3.568433,-0.450986
2023-01-01 00:03:00,9.713153,7.086587,-1.469676,5.505332,0.231826,2.303585,-1.082641,-2.656692,1.290334,-11.724882,...,-7.007576,-1.550226,-2.695628,-5.792614,-0.638659,-1.712024,2.617396,-3.26492,-3.568433,-0.450986
2023-01-01 00:04:00,9.713258,7.087006,-1.467071,5.506144,0.231270,2.302585,-1.082936,-2.656407,1.289783,-11.723646,...,-7.007576,-1.550226,-2.697110,-5.792614,-0.638659,-1.712024,2.617396,-3.26492,-3.568433,-0.454130


,ticker
0,BTCUSDT
1,ETHUSDT
2,GMTUSDT
3,BNBUSDT
4,LUNAUSDT


In [ ]:
# Load PRVM dates and define prediction dates

loaded = np.load(PRVM_NPZ_PATH, allow_pickle=True)

days = pd.to_datetime(loaded["days"])
Gamma_shape = loaded["Gamma"].shape

n = 365
T = len(days)
pred_dates = days[n:T]

print("Gamma shape:", Gamma_shape)
print("days:", days.min(), "~", days.max(), "len=", len(days))
print("n:", n)
print("pred_dates:", pred_dates.min(), "~", pred_dates.max(), "len=", len(pred_dates))


pd.DataFrame({"date": pred_dates}).to_csv(
    PRVM_OUTPUT_DIR / "pred_dates_crypto_2023_2025.csv",
    index=False
)

Gamma shape: (1095, 200, 200)
days: 2023-01-01 00:00:00 ~ 2025-12-31 00:00:00 len= 1095
n: 365
pred_dates: 2024-01-02 00:00:00 ~ 2025-12-31 00:00:00 len= 730


In [ ]:
# Build daily 10-minute log-return tensor

def make_daily_10min_returns_from_log_price(
    log_price_df: pd.DataFrame,
    pred_dates,
    tickers=None,
    freq="10min",
    expected_bins_per_day=144,
    fill_missing_with_zero=True,
    dtype=np.float32,
):

    df = log_price_df.copy()
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    if tickers is not None:
        df = df[tickers]

    # 1-minute log returns. The first minute of a day uses the previous minute if available.
    ret_1min = df.diff()

    pred_dates = pd.to_datetime(pred_dates)
    p = df.shape[1]
    out = np.zeros((len(pred_dates), expected_bins_per_day, p), dtype=dtype)
    missing_counts = []

    for idx, d in enumerate(tqdm(pred_dates, desc="Building daily 10-min returns")):
        day_start = pd.Timestamp(d.date())
        day_end = day_start + pd.Timedelta(days=1)

        g = ret_1min.loc[(ret_1min.index >= day_start) & (ret_1min.index < day_end)]

        # Sum 1-minute log returns inside each 10-minute bin.
        g10 = g.resample(freq).sum(min_count=1)

        # Force exactly 144 bins: 00:00, 00:10, ..., 23:50
        expected_index = day_start + pd.to_timedelta(np.arange(0, 1440, 10), unit="min")
        g10 = g10.reindex(expected_index)

        missing_counts.append(int(g10.isna().all(axis=1).sum()))

        if fill_missing_with_zero:
            g10 = g10.fillna(0.0)

        arr = g10.to_numpy(dtype=np.float64)
        arr = np.nan_to_num(arr, nan=0.0)

        out[idx] = arr.astype(dtype)

    missing_counts = np.array(missing_counts)
    return out, missing_counts


returns_10min_pred, missing_10min_bins = make_daily_10min_returns_from_log_price(
    log_price_df=log_price,
    pred_dates=pred_dates,
    tickers=tickers,
    dtype=np.float32,
)

print("returns_10min_pred shape:", returns_10min_pred.shape)
print("missing bins summary:")
print(pd.Series(missing_10min_bins).describe())


Building daily 10-min returns: 100%|██████████| 730/730 [00:05<00:00, 131.24it/s]


returns_10min_pred shape: (730, 144, 200)
missing bins summary:
count    730.0
mean       0.0
std        0.0
min        0.0
25%        0.0
50%        0.0
75%        0.0
max        0.0
dtype: float64


In [ ]:
# Save outputs for the original FIVAR portfolio-risk notebook

returns_path = PRVM_OUTPUT_DIR / "returns_10min_pred_crypto.npy"
# missing_path = PRVM_OUTPUT_DIR / "returns_10min_missing_bins_crypto.npy"
# tickers_path = PRVM_OUTPUT_DIR / "returns_10min_tickers_crypto.csv"

np.save(returns_path, returns_10min_pred)
# np.save(missing_path, missing_10min_bins)
# pd.DataFrame({"ticker": tickers}).to_csv(tickers_path, index=False)

print("Saved:", returns_path)
# print("Saved:", missing_path)
# print("Saved:", tickers_path)

# reload check
check = np.load(returns_path)
print("Reloaded shape:", check.shape)
print("First day shape:", check[0].shape)


## Files created

- `returns_10min_pred_crypto.npy`: shape `(T-n, 144, 200)`, use this in portfolio risk evaluation.
- `pred_dates_crypto_2023_2025.csv`: dates aligned with prediction covariance matrices.
- `returns_10min_tickers_crypto.csv`: ticker order used in the return tensor.
- `returns_10min_missing_bins_crypto.npy`: diagnostic count of fully missing 10-minute bins per prediction day.
